# Systematic Misprice Backtest — When Do Edges Appear?

The dataset survey found 25 misprices across 23/141 movies with median 57¢ edge in the last 24h before close. This notebook traces **when** those edges appeared and how long they persisted.

**Core method:** At each point in time, compute worst-case and best-case final scores from accumulated reviews. An outcome is **locked** when remaining reviews cannot change the resolution. Compare locked outcomes to market prices to find edges.

**Plan:** `plans/plan_misprice_backtest.md` · **Backlog:** §2.8

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from glob import glob
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'figure.dpi': 110,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

ROOT = Path('.').resolve().parent  # notebooks/ → project root
print(f"Project root: {ROOT}")

## 1. Load Data

In [ ]:
# ── Movies index ──────────────────────────────────────────────────────
mi = pd.read_csv(ROOT / 'movies_index.csv')

mi['volume'] = mi['Trading Volume ($)'].str.replace(r'[\$,]', '', regex=True).astype(float)

for col in ['Embargo Lift Date', 'Bet Open Date', 'Bet Close Date']:
    mi[col] = pd.to_datetime(mi[col], utc=True)

# Score range (fractions like 0.8750-0.9050)
mi[['score_low', 'score_high']] = mi['Tomatometer Score Range Bet Close'].str.split('-', expand=True).astype(float)
mi['score_low_pct'] = mi['score_low'] * 100
mi['score_high_pct'] = mi['score_high'] * 100
mi['score_mid_pct'] = (mi['score_low_pct'] + mi['score_high_pct']) / 2

# Review count range
mi[['reviews_low', 'reviews_high']] = mi['Total Reviews Bet Close'].str.split('-', expand=True).astype(float)
mi['reviews_T'] = mi['reviews_high']  # upper bound for conservative bounds

mi['embargo_to_close_days'] = (mi['Bet Close Date'] - mi['Embargo Lift Date']).dt.days

print(f"Movies: {len(mi)}")
print(f"Review count T (upper bound) range: {mi['reviews_T'].min():.0f} – {mi['reviews_T'].max():.0f}")
mi[['Slug', 'volume', 'Bet Close Date', 'score_low_pct', 'score_high_pct', 'reviews_T']].head(3)

In [ ]:
# ── Reviews ───────────────────────────────────────────────────────────
reviews = pd.read_csv(ROOT / 'reviews.csv')
reviews['estimated_timestamp'] = pd.to_datetime(reviews['estimated_timestamp'], utc=True, format='ISO8601')
reviews['is_fresh'] = (reviews['tomatometer_sentiment'] == 'positive').astype(int)
reviews['review_date'] = reviews['estimated_timestamp'].dt.normalize()  # midnight UTC

print(f"Reviews: {len(reviews):,}")
print(f"Movies in reviews: {reviews['movie_slug'].nunique()}")
print(f"Date range: {reviews['review_date'].min().date()} – {reviews['review_date'].max().date()}")

In [ ]:
# ── Price histories (hour-level) ──────────────────────────────────────
def load_price_csv(movie_slug, freq='hour'):
    d = ROOT / 'rt-price-histories' / movie_slug
    if not d.exists():
        return None
    matches = list(d.glob(f'*-{freq}.csv'))
    if not matches:
        return None
    df = pd.read_csv(matches[0], parse_dates=['timestamp'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df = df.set_index('timestamp').sort_index()
    return df

price_data = {}
threshold_cols = {}
for slug in mi['Slug']:
    df = load_price_csv(slug, 'hour')
    if df is not None:
        price_data[slug] = df
        threshold_cols[slug] = [c for c in df.columns if c.startswith('Above')]

print(f"Loaded price histories for {len(price_data)} / {len(mi)} movies")

## 2. Cumulative Review Counts + Score Bounds Per Movie

In [ ]:
def build_cumulative_series(slug, reviews_df, total_reviews_T):
    """Build daily cumulative review counts and score bounds for a movie.
    
    Returns DataFrame with columns:
        review_date, cum_positive (K), cum_total (N), implied_score_pct,
        best_case_pct, worst_case_pct
    """
    movie_revs = reviews_df[reviews_df['movie_slug'] == slug].copy()
    if movie_revs.empty:
        return None
    
    # Daily aggregation: count reviews and fresh reviews per date
    daily = movie_revs.groupby('review_date').agg(
        n_reviews=('is_fresh', 'count'),
        n_fresh=('is_fresh', 'sum'),
    ).sort_index()
    
    # Cumulative sums
    daily['cum_total'] = daily['n_reviews'].cumsum()        # N
    daily['cum_positive'] = daily['n_fresh'].cumsum()       # K
    
    # Use max(T, N) as effective total — if more reviews exist than T predicted,
    # the remaining count is 0 and bounds collapse to the point estimate.
    T = total_reviews_T
    daily['effective_T'] = daily['cum_total'].clip(lower=0).combine(pd.Series(T, index=daily.index), max)
    daily['reviews_remaining'] = (daily['effective_T'] - daily['cum_total']).clip(lower=0)
    
    daily['implied_score_pct'] = (daily['cum_positive'] / daily['cum_total']) * 100
    daily['best_case_pct'] = ((daily['cum_positive'] + daily['reviews_remaining']) / daily['effective_T']) * 100
    daily['worst_case_pct'] = (daily['cum_positive'] / daily['effective_T']) * 100
    
    return daily.reset_index()

# Build for all movies
cum_series = {}
for _, row in mi.iterrows():
    slug = row['Slug']
    T = row['reviews_T']
    if pd.isna(T) or T == 0:
        continue
    series = build_cumulative_series(slug, reviews, T)
    if series is not None:
        cum_series[slug] = series

print(f"Built cumulative series for {len(cum_series)} movies")

# Sanity check: show bounds for a sample movie at last date
sample_slug = list(cum_series.keys())[0]
sample = cum_series[sample_slug]
print(f"\nSample: {sample_slug} (last row)")
print(sample.iloc[-1][['cum_total', 'cum_positive', 'implied_score_pct', 'best_case_pct', 'worst_case_pct', 'reviews_remaining']].to_string())

# Check for movies where reviews exceeded T
exceeded_T = []
for _, row in mi.iterrows():
    slug = row['Slug']
    if slug not in cum_series:
        continue
    final = cum_series[slug].iloc[-1]
    if final['cum_total'] > row['reviews_T']:
        exceeded_T.append((slug, int(final['cum_total']), int(row['reviews_T'])))
if exceeded_T:
    print(f"\n{len(exceeded_T)} movies had more reviews than T (upper bound). Bounds clamped to point estimate:")
    for s, actual, t in exceeded_T[:5]:
        print(f"  {s}: {actual} actual vs {t} T")

In [ ]:
# Sanity check: final implied scores should be within movies_index score range
convergence_issues = []
for _, row in mi.iterrows():
    slug = row['Slug']
    if slug not in cum_series:
        continue
    final = cum_series[slug].iloc[-1]
    score = final['implied_score_pct']
    if score < row['score_low_pct'] - 1 or score > row['score_high_pct'] + 1:
        convergence_issues.append({
            'slug': slug,
            'final_implied': round(score, 1),
            'index_range': f"{row['score_low_pct']:.1f}–{row['score_high_pct']:.1f}",
            'cum_reviews': int(final['cum_total']),
            'T': int(row['reviews_T']),
        })

if convergence_issues:
    print(f"WARNING: {len(convergence_issues)} movies have final score outside index range (±1pp tolerance):")
    display(pd.DataFrame(convergence_issues))
else:
    print("All final implied scores within index range (+/- 1pp). OK.")

## 3. Edge Time Series — Locked Outcomes vs. Market Prices

In [ ]:
def get_threshold_value(col_name):
    return int(col_name.split()[-1])

def resolve_threshold(threshold_val, score_low_pct, score_high_pct):
    """Actual resolution from movies_index score range."""
    displayed_low = round(score_low_pct)
    displayed_high = round(score_high_pct)
    needed = threshold_val + 1
    if displayed_low >= needed:
        return 'Yes'
    elif displayed_high < needed:
        return 'No'
    else:
        return 'Ambiguous'

def check_lock(best_case_pct, worst_case_pct, threshold_val):
    """Check if outcome is locked from review bounds.
    Returns 'Yes', 'No', or None (uncertain)."""
    needed = threshold_val + 1
    if round(worst_case_pct) >= needed:
        return 'Yes'   # locked above threshold
    elif round(best_case_pct) < needed:
        return 'No'    # locked below threshold
    return None         # uncertain

print("Helper functions defined.")

In [ ]:
# Build the full edge time series
edge_records = []
lock_mismatches = 0
lock_mismatch_slugs = set()

for _, mrow in mi.iterrows():
    slug = mrow['Slug']
    if slug not in cum_series or slug not in price_data:
        continue
    
    series = cum_series[slug]
    prices = price_data[slug]
    bet_close = mrow['Bet Close Date']
    score_low_pct = mrow['score_low_pct']
    score_high_pct = mrow['score_high_pct']
    
    # Forward-fill prices so we can look up any timestamp
    prices_ffill = prices.ffill()
    
    for col in threshold_cols.get(slug, []):
        thresh_val = get_threshold_value(col)
        actual_resolution = resolve_threshold(thresh_val, score_low_pct, score_high_pct)
        if actual_resolution == 'Ambiguous':
            continue
        
        for _, srow in series.iterrows():
            review_date = srow['review_date']
            hours_before_close = (bet_close - review_date).total_seconds() / 3600
            
            if hours_before_close < 0:
                continue
            
            # Look up market price: last available on or before this date
            mask = prices_ffill.index <= review_date + pd.Timedelta(hours=23, minutes=59)
            if mask.any() and col in prices_ffill.columns:
                price_at_date = prices_ffill.loc[mask, col].iloc[-1]
            else:
                price_at_date = np.nan
            
            if pd.isna(price_at_date):
                continue
            
            # Check lock status
            lock = check_lock(srow['best_case_pct'], srow['worst_case_pct'], thresh_val)
            
            # Only compute edge when lock agrees with actual resolution
            if lock is not None:
                if lock != actual_resolution:
                    lock_mismatches += 1
                    lock_mismatch_slugs.add(slug)
                    edge = np.nan  # lock is wrong — don't trust it
                elif lock == 'Yes':
                    edge = 100 - price_at_date
                else:
                    edge = price_at_date
            else:
                edge = np.nan
            
            edge_records.append({
                'slug': slug,
                'threshold': thresh_val,
                'actual_resolution': actual_resolution,
                'review_date': review_date,
                'hours_before_close': hours_before_close,
                'cum_reviews': srow['cum_total'],
                'cum_positive': srow['cum_positive'],
                'reviews_remaining': srow['reviews_remaining'],
                'implied_score_pct': srow['implied_score_pct'],
                'best_case_pct': srow['best_case_pct'],
                'worst_case_pct': srow['worst_case_pct'],
                'lock': lock,
                'lock_correct': lock == actual_resolution if lock is not None else None,
                'market_price': price_at_date,
                'edge_cents': edge,
                'volume': mrow['volume'],
            })

edges = pd.DataFrame(edge_records)
print(f"Edge time series: {len(edges):,} rows")
print(f"  Locked observations: {edges['lock'].notna().sum():,}")
print(f"  Uncertain observations: {edges['lock'].isna().sum():,}")
print(f"  Lock/resolution mismatches: {lock_mismatches} (excluded from edge computation)")
if lock_mismatch_slugs:
    print(f"  Mismatched movies ({len(lock_mismatch_slugs)}): review final score ≠ index score range")
    for s in sorted(lock_mismatch_slugs)[:8]:
        print(f"    - {s}")

In [ ]:
# Filter to locked observations with meaningful positive edge
locked = edges[edges['lock'].notna()].copy()
locked_with_edge = locked[locked['edge_cents'] > 0].copy()

# Summary at different edge thresholds
print(f"Locked observations: {len(locked):,}")
print(f"  With any positive edge (>0¢):  {len(locked_with_edge):,}")
print(f"  With edge > 5¢:   {(locked_with_edge['edge_cents'] > 5).sum():,}")
print(f"  With edge > 10¢:  {(locked_with_edge['edge_cents'] > 10).sum():,}")
print(f"  With edge > 20¢:  {(locked_with_edge['edge_cents'] > 20).sum():,}")
print(f"  With edge > 50¢:  {(locked_with_edge['edge_cents'] > 50).sum():,}")

# Use 5¢ as minimum meaningful edge for episode analysis
# (below 5¢ the market is basically correct given fees)
MIN_EDGE = 5
mispriced = locked_with_edge[locked_with_edge['edge_cents'] > MIN_EDGE].copy()

print(f"\nUsing >{MIN_EDGE}¢ threshold for episode analysis:")
print(f"  Mispriced observations: {len(mispriced):,}")
print(f"  Movies: {mispriced['slug'].nunique()}")
print(f"  Unique movie×threshold pairs: {mispriced.groupby(['slug', 'threshold']).ngroups}")
print(f"  Median edge: {mispriced['edge_cents'].median():.1f}¢")
print(f"  Mean edge:   {mispriced['edge_cents'].mean():.1f}¢")
print(f"  Max edge:    {mispriced['edge_cents'].max():.1f}¢")

## 4. Misprice Episodes — When Did Edges Appear and Persist?

In [ ]:
# For each movie × threshold pair that has mispriced observations,
# build an episode from lock time through close.
episode_records = []

for (slug, thresh), grp in mispriced.groupby(['slug', 'threshold']):
    grp = grp.sort_values('hours_before_close', ascending=False)  # earliest first
    
    # Also get the full locked history for context (including low-edge observations)
    full_locked = locked[(locked['slug'] == slug) & (locked['threshold'] == thresh)].sort_values(
        'hours_before_close', ascending=False)
    first_lock = full_locked.iloc[0]
    
    episode_records.append({
        'slug': slug,
        'threshold': thresh,
        'actual_resolution': first_lock['actual_resolution'],
        'lock_direction': first_lock['lock'],
        'volume': first_lock['volume'],
        # When outcome locked
        'lock_hours_before_close': first_lock['hours_before_close'],
        'lock_cum_reviews': first_lock['cum_reviews'],
        'lock_reviews_remaining': first_lock['reviews_remaining'],
        # Edge timing (>5¢ observations only)
        'first_edge_hours_before_close': grp['hours_before_close'].max(),
        'last_edge_hours_before_close': grp['hours_before_close'].min(),
        'edge_duration_hours': grp['hours_before_close'].max() - grp['hours_before_close'].min(),
        # Edge size
        'max_edge_cents': grp['edge_cents'].max(),
        'mean_edge_cents': grp['edge_cents'].mean(),
        'last_edge_cents': grp.sort_values('hours_before_close').iloc[0]['edge_cents'],
        # Activity
        'n_edge_observations': len(grp),
        'n_total_locked_obs': len(full_locked),
    })

episodes = pd.DataFrame(episode_records)
print(f"Misprice episodes (>{MIN_EDGE}¢): {len(episodes)}")
print(f"  Across {episodes['slug'].nunique()} movies")
print(f"\nEdge first appeared (hours before close):")
print(f"  Median: {episodes['first_edge_hours_before_close'].median():.0f}h")
print(f"  Mean:   {episodes['first_edge_hours_before_close'].mean():.0f}h")
print(f"  Max:    {episodes['first_edge_hours_before_close'].max():.0f}h")
print(f"\nEdge duration (first >5¢ obs to last >5¢ obs):")
print(f"  Median: {episodes['edge_duration_hours'].median():.0f}h")
print(f"  Mean:   {episodes['edge_duration_hours'].mean():.0f}h")
print(f"\nMax edge per episode:")
print(f"  Median: {episodes['max_edge_cents'].median():.1f}¢")
print(f"  Mean:   {episodes['max_edge_cents'].mean():.1f}¢")

In [ ]:
# Top episodes by edge size
top_episodes = episodes.nlargest(20, 'max_edge_cents')[
    ['slug', 'threshold', 'lock_direction', 'max_edge_cents', 'mean_edge_cents',
     'first_edge_hours_before_close', 'edge_duration_hours',
     'lock_cum_reviews', 'lock_reviews_remaining', 'volume']
].copy()
top_episodes['volume'] = top_episodes['volume'].apply(lambda x: f"${x/1e6:.2f}M")
top_episodes['max_edge_cents'] = top_episodes['max_edge_cents'].round(1)
top_episodes['mean_edge_cents'] = top_episodes['mean_edge_cents'].round(1)
top_episodes['first_edge_hours_before_close'] = top_episodes['first_edge_hours_before_close'].round(0)
top_episodes['edge_duration_hours'] = top_episodes['edge_duration_hours'].round(0)
top_episodes.columns = ['Movie', 'Threshold', 'Lock', 'Max Edge (¢)', 'Mean Edge (¢)',
                         'First Edge (h before close)', 'Duration (h)',
                         'Reviews at Lock', 'Remaining at Lock', 'Volume']
display(top_episodes.reset_index(drop=True))

## 5. Segmentation by Volume Quartile

In [ ]:
# Assign volume quartiles
mi['vol_quartile'] = pd.qcut(mi['volume'], 4, labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)'])
vol_q_map = mi.set_index('Slug')['vol_quartile'].to_dict()

episodes['vol_quartile'] = episodes['slug'].map(vol_q_map)

# Quartile summary
q_summary = episodes.groupby('vol_quartile').agg(
    n_episodes=('slug', 'count'),
    n_movies=('slug', 'nunique'),
    median_max_edge=('max_edge_cents', 'median'),
    mean_max_edge=('max_edge_cents', 'mean'),
    median_first_edge_h=('first_edge_hours_before_close', 'median'),
    median_duration_h=('edge_duration_hours', 'median'),
    median_lock_reviews=('lock_cum_reviews', 'median'),
).reset_index()

# Add total movies per quartile for context
movies_per_q = mi.groupby('vol_quartile')['Slug'].count().reset_index()
movies_per_q.columns = ['vol_quartile', 'total_movies']
q_summary = q_summary.merge(movies_per_q, on='vol_quartile')
q_summary['pct_movies_with_edge'] = (q_summary['n_movies'] / q_summary['total_movies'] * 100).round(1)

print("Misprice episodes by volume quartile:")
display(q_summary)

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. When do edges first appear? (hours before close)
ax = axes[0, 0]
ax.hist(episodes['first_edge_hours_before_close'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
ax.set_xlabel('Hours Before Close When Edge First Appeared')
ax.set_ylabel('Count (episodes)')
ax.set_title('When Do Locked Edges First Appear?')
ax.axvline(episodes['first_edge_hours_before_close'].median(), color='red', ls='--',
           label=f"Median: {episodes['first_edge_hours_before_close'].median():.0f}h")
ax.axvline(24, color='orange', ls=':', label='24h before close')
ax.legend(fontsize=8)

# 2. Edge size distribution
ax = axes[0, 1]
ax.hist(episodes['max_edge_cents'], bins=25, edgecolor='black', alpha=0.7, color='gold')
ax.set_xlabel('Max Edge per Episode (¢)')
ax.set_ylabel('Count')
ax.set_title('Edge Size Distribution')
ax.axvline(episodes['max_edge_cents'].median(), color='red', ls='--',
           label=f"Median: {episodes['max_edge_cents'].median():.0f}¢")
ax.legend(fontsize=8)

# 3. Edge duration
ax = axes[1, 0]
ax.hist(episodes['edge_duration_hours'], bins=30, edgecolor='black', alpha=0.7, color='mediumseagreen')
ax.set_xlabel('Edge Duration (hours)')
ax.set_ylabel('Count')
ax.set_title('How Long Do Edges Persist?')
ax.axvline(episodes['edge_duration_hours'].median(), color='red', ls='--',
           label=f"Median: {episodes['edge_duration_hours'].median():.0f}h")
ax.legend(fontsize=8)

# 4. Edge by volume quartile (box plot)
ax = axes[1, 1]
quartile_order = ['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)']
box_data = [episodes[episodes['vol_quartile'] == q]['max_edge_cents'].values for q in quartile_order]
bp = ax.boxplot(box_data, labels=quartile_order, patch_artist=True)
colors = ['#AEC7E8', '#98DF8A', '#FFBB78', '#FF9896']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_xlabel('Volume Quartile')
ax.set_ylabel('Max Edge (¢)')
ax.set_title('Edge Size by Volume Quartile')

plt.tight_layout()
plt.show()

In [ ]:
# Scatter: max edge vs. hours before close when edge first appeared
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
scatter_colors = {'Q1 (lowest)': '#1f77b4', 'Q2': '#2ca02c', 'Q3': '#ff7f0e', 'Q4 (highest)': '#d62728'}
for q in quartile_order:
    subset = episodes[episodes['vol_quartile'] == q]
    ax.scatter(subset['first_edge_hours_before_close'], subset['max_edge_cents'],
               alpha=0.6, s=40, label=q, color=scatter_colors[q], edgecolor='k', linewidth=0.3)
ax.set_xlabel('Hours Before Close When Edge First Appeared')
ax.set_ylabel('Max Edge (¢)')
ax.set_title('Edge Size vs. Timing')
ax.axvline(24, color='gray', ls=':', alpha=0.5)
ax.legend(fontsize=8, title='Volume Q')

# Edge vs. review fraction when lock happened
ax = axes[1]
# What fraction of total reviews were in when the edge first appeared?
episodes['lock_review_fraction'] = episodes['lock_cum_reviews'] / (episodes['lock_cum_reviews'] + episodes['lock_reviews_remaining'])
ax.scatter(episodes['lock_review_fraction'] * 100, episodes['max_edge_cents'],
           alpha=0.5, s=40, c='steelblue', edgecolor='k', linewidth=0.3)
ax.set_xlabel('% of Total Reviews In When Outcome Locked')
ax.set_ylabel('Max Edge (¢)')
ax.set_title('Edge Size vs. Review Completeness at Lock')
ax.axvline(90, color='gray', ls=':', alpha=0.5, label='90% complete')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Timeline plots for the top 6 highest-edge movies
top_movie_slugs = episodes.nlargest(6, 'max_edge_cents')['slug'].unique()[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes_flat = axes.flatten()

for i, slug in enumerate(top_movie_slugs):
    ax = axes_flat[i]
    movie_mispriced = mispriced[mispriced['slug'] == slug]
    
    if movie_mispriced.empty:
        ax.set_title(slug.replace('_', ' ').title())
        continue
    
    # Plot edge over time for each threshold
    for thresh in sorted(movie_mispriced['threshold'].unique()):
        t_data = movie_mispriced[movie_mispriced['threshold'] == thresh].sort_values('hours_before_close')
        ax.plot(t_data['hours_before_close'], t_data['edge_cents'], 
                marker='o', markersize=4, label=f'Above {thresh}', alpha=0.8)
    
    ax.set_xlabel('Hours Before Close')
    ax.set_ylabel('Edge (¢)')
    ax.set_title(slug.replace('_', ' ').title(), fontsize=10)
    ax.invert_xaxis()
    ax.axhline(0, color='gray', ls='-', alpha=0.3)
    ax.legend(fontsize=7, loc='best')

# Hide unused axes
for j in range(len(top_movie_slugs), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle(f'Edge Evolution Over Time — Top Mispriced Movies (>{MIN_EDGE}¢)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Persistence — Do Edges Self-Correct or Persist to Close?

In [ ]:
# Did the edge persist until close (last observation within 48h of close)?
episodes['persisted_to_close'] = episodes['last_edge_hours_before_close'] <= 48
episodes['appeared_early'] = episodes['first_edge_hours_before_close'] > 48

early_episodes = episodes[episodes['appeared_early']]
print(f"Episodes where edge appeared >48h before close: {len(early_episodes)}")
if len(early_episodes) > 0:
    print(f"  Of those, persisted to within 48h of close: {early_episodes['persisted_to_close'].sum()} "
          f"({early_episodes['persisted_to_close'].mean()*100:.0f}%)")
    print(f"  Self-corrected before 48h: {(~early_episodes['persisted_to_close']).sum()} "
          f"({(~early_episodes['persisted_to_close']).mean()*100:.0f}%)")

# All episodes: did edge persist to close?
print(f"\nAll episodes:")
print(f"  Edge present within 48h of close: {episodes['persisted_to_close'].sum()} "
      f"({episodes['persisted_to_close'].mean()*100:.0f}%)")
print(f"  Edge resolved before 48h: {(~episodes['persisted_to_close']).sum()} "
      f"({(~episodes['persisted_to_close']).mean()*100:.0f}%)")

# For episodes that persisted: what was the edge at the last observation?
persisted = episodes[episodes['persisted_to_close']]
if len(persisted) > 0:
    print(f"\nPersisted episodes — edge at last observation:")
    print(f"  Median: {persisted['last_edge_cents'].median():.1f}¢")
    print(f"  Mean:   {persisted['last_edge_cents'].mean():.1f}¢")

## 8. Cross-Check with Dataset Survey

The survey found 25 misprices (>30¢, last 24h) across 23 movies. The bounds approach is more conservative — verify the big ones still appear.

In [ ]:
# The survey's top misprices (from brainstorm_dataset_survey_findings.md):
# Joker: Folie a Deux — Above 35/40/45, 97¢ edge
# The Wild Robot — Above 97, 88¢ edge  
# Heart Eyes — Above 80, 81¢ edge
# Wolf Man 2025 — Above 52, 81¢ edge
# Disney's Snow White — Above 43, 61¢ edge
# A Minecraft Movie — Above 48, 61¢ edge
# Mickey 17 — Above 78, 51¢ edge

survey_top = [
    ('joker_folie_a_deux', [35, 40, 45]),
    ('the_wild_robot', [97]),
    ('heart_eyes', [80]),
    ('wolf_man_2025', [52]),
    ('disneys_snow_white', [43]),
    ('a_minecraft_movie', [48]),
    ('mickey_17', [78]),
]

print("Cross-check: survey's biggest misprices vs. backtest episodes\n")
for slug, thresholds in survey_top:
    for t in thresholds:
        match = episodes[(episodes['slug'] == slug) & (episodes['threshold'] == t)]
        if not match.empty:
            row = match.iloc[0]
            print(f"  {slug} Above {t}: max_edge={row['max_edge_cents']:.0f}¢, "
                  f"first_edge={row['first_edge_hours_before_close']:.0f}h before close")
        else:
            # Check if it's in the locked data at all
            locked_match = locked[(locked['slug'] == slug) & (locked['threshold'] == t)]
            if locked_match.empty:
                print(f"  {slug} Above {t}: NOT LOCKED (threshold never locked from bounds)")
            else:
                print(f"  {slug} Above {t}: Locked but no positive edge (market correctly priced)")

## 9. Summary

In [ ]:
print("=" * 70)
print("SYSTEMATIC MISPRICE BACKTEST — SUMMARY")
print("=" * 70)
print(f"""
Method: Worst-case/best-case bounds from accumulated reviews.
        Edge counted only when remaining reviews CANNOT change outcome.
        T = upper bound of review count range (conservative).
        Minimum edge threshold: >{MIN_EDGE}¢ (below this, market is ~correct).

Movies analyzed:     {len(cum_series)} (with both reviews + price data)
Misprice episodes:   {len(episodes)} (across {episodes['slug'].nunique()} movies)

TIMING:
  Edge first appears:  median {episodes['first_edge_hours_before_close'].median():.0f}h before close
  Edge duration:       median {episodes['edge_duration_hours'].median():.0f}h

EDGE SIZE:
  Max edge per episode: median {episodes['max_edge_cents'].median():.0f}¢, mean {episodes['max_edge_cents'].mean():.0f}¢
  Edges > 20¢:         {(episodes['max_edge_cents'] > 20).sum()}
  Edges > 50¢:         {(episodes['max_edge_cents'] > 50).sum()}

PERSISTENCE:
  Persisted to within 48h of close: {episodes['persisted_to_close'].sum()}/{len(episodes)} ({episodes['persisted_to_close'].mean()*100:.0f}%)

VOLUME QUARTILE BREAKDOWN:""")

for _, qrow in q_summary.iterrows():
    print(f"  {qrow['vol_quartile']}: {qrow['n_episodes']} episodes across "
          f"{qrow['n_movies']}/{qrow['total_movies']} movies, "
          f"median max edge {qrow['median_max_edge']:.0f}¢")